## Prompting Lab – Module 101

# TASK 1 – CLASSIFICATION

## 1.1 Zero‑shot prompt

**Prompt:**

> Categorize each unique LICENSESTATUS value from the license dataset as either operational or non-operational, and explain your reasoning.

**Output:**

### Unique values found in `license.csv`

The following 11 distinct values appear in the `LICENSESTATUS` field across all ~45,000 records:

| LICENSESTATUS | Record count |
|---|---|
| ACTIVE | 42,665 |
| CANCELLED_NOT_RENEWED | 1,163 |
| PENDING_RENEWAL | 632 |
| TERMINATED | 475 |
| BY REQUEST | 337 |
| EXPIRED | 68 |
| HOLD_TSD | 24 |
| TERMINATED DECEASED | 6 |
| CANCELLED_BY_CUST_REQ | 6 |
| ENTERED | 4 |
| CANCELLED | 3 |

### Categorization and reasoning

| Status | Category | Reasoning |
|---|---|---|
| **ACTIVE** | Operational | The elevator holds a current, valid license. It is legally permitted to operate. |
| **PENDING_RENEWAL** | Operational | The license is in the renewal process but has not yet lapsed. The device is still licensed and presumed in service during this window. |
| **BY REQUEST** | Operational | The license was placed on hold at the owner's request — typically while a building is closed for renovation or temporarily out of use. The device is intact and can return to service. This is a managed suspension, not a termination. |
| **HOLD_TSD** | Operational | TSD (Technical Standards and Safety Division) has placed the license on administrative hold, usually pending paperwork or compliance follow-up. The device has not been decommissioned and is expected to return to active status. |
| **ENTERED** | Operational | A data-entry or intake state — the record has been created but not yet fully processed. The device has not been rejected or terminated. |
| **EXPIRED** | Non-operational | The license passed its expiry date without renewal. Operating an unlicensed elevator is a regulatory violation in Ontario; the device must not run until the license is renewed. |
| **CANCELLED_NOT_RENEWED** | Non-operational | The owner did not renew at expiry and the license was formally cancelled. Distinct from EXPIRED in that the cancellation is finalized rather than still recoverable by late renewal. |
| **CANCELLED_BY_CUST_REQ** | Non-operational | The license was explicitly cancelled at the licenseholder's request — typically when a device is removed, replaced, or a property changes ownership. The device is no longer in service. |
| **CANCELLED** | Non-operational | General cancellation state. A cancelled license means the device is not legally permitted to operate. |
| **TERMINATED** | Non-operational | The license was revoked or the device was decommissioned. Stronger than cancellation; the license relationship is permanently ended. |
| **TERMINATED DECEASED** | Non-operational | The licenseholder is deceased and the license was closed as a result. The device cannot legally operate until a new licenseholder obtains a new license. |

### Summary

| Category | Statuses included | Record count | Share |
|---|---|---|---|
| **Operational** | ACTIVE, PENDING_RENEWAL, BY REQUEST, HOLD_TSD, ENTERED | 43,662 | 96.2% |
| **Non-operational** | EXPIRED, CANCELLED_NOT_RENEWED, CANCELLED_BY_CUST_REQ, CANCELLED, TERMINATED, TERMINATED DECEASED | 1,721 | 3.8% |

> **Note for the dashboard:** The current specification defines Card 2 (Active Elevators) using only `LICENSESTATUS = "ACTIVE"`. If the operations manager also wants to count devices that are temporarily suspended but still viable (BY REQUEST, PENDING_RENEWAL, HOLD_TSD), the card definition and its calculation rule would need to be revisited.

## 1.2 Few‑shot prompt

# *Prompt* : 
Categorize each LICENSESTATUS value as either operational or non-operational.

Examples:
- ACTIVE → Operational (the elevator is licensed and allowed to operate)
- CANCELLED_NOT_RENEWED → Non-operational (the license is no longer valid)

Now categorize all remaining LICENSESTATUS values in the dataset and explain your reasoning.

# *Output* : 

### Examples given in the prompt (anchors)

| Status | Category | Reason given in prompt |
|---|---|---|
| ACTIVE | Operational | The elevator is licensed and allowed to operate |
| CANCELLED_NOT_RENEWED | Non-operational | The license is no longer valid |

These two examples define the classification boundary: a status is **Operational** if the device retains a valid or recoverable license; **Non-operational** if the license has lapsed, been cancelled, or terminated.

### Remaining 9 values categorized

| Status | Category | Reasoning |
|---|---|---|
| **PENDING_RENEWAL** | Operational | The license is actively being renewed and has not yet lapsed. By the same logic as ACTIVE, the elevator retains valid standing during the renewal window. |
| **BY REQUEST** | Operational | The owner voluntarily suspended the license — a managed hold, not a lapse. The device can return to service when the hold is lifted, placing it closer to ACTIVE than to CANCELLED. |
| **HOLD_TSD** | Operational | An administrative hold by the regulator (TSSA), typically pending paperwork. The license has not been revoked; the device is expected to return to active status once resolved. |
| **ENTERED** | Operational | A data-entry or intake state. The record exists but processing is incomplete. No rejection or termination has occurred, so the device is not yet confirmed non-operational. |
| **EXPIRED** | Non-operational | Same logic as CANCELLED_NOT_RENEWED: the license is no longer valid. The elevator cannot legally operate until the license is renewed. |
| **CANCELLED_BY_CUST_REQ** | Non-operational | The licenseholder explicitly requested cancellation — typically because the device was removed or decommissioned. The license is no longer valid by the same standard as CANCELLED_NOT_RENEWED. |
| **CANCELLED** | Non-operational | General cancellation state. The license is no longer valid. Same category as CANCELLED_NOT_RENEWED. |
| **TERMINATED** | Non-operational | The license was permanently revoked or the device was decommissioned. A stronger form of non-validity than cancellation; the relationship is ended. |
| **TERMINATED DECEASED** | Non-operational | The license closed due to the licenseholder's death. The device cannot operate legally until a new licenseholder is registered — equivalent to having no valid license. |

### Full categorization summary

| Category | Statuses | Record count | Share |
|---|---|---|---|
| **Operational** | ACTIVE, PENDING_RENEWAL, BY REQUEST, HOLD_TSD, ENTERED | 43,662 | 96.2% |
| **Non-operational** | EXPIRED, CANCELLED_NOT_RENEWED, CANCELLED_BY_CUST_REQ, CANCELLED, TERMINATED, TERMINATED DECEASED | 1,721 | 3.8% |



## 1.3 Chain‑of‑thought prompt

# *Prompt*: 
Categorize each LICENSESTATUS value as operational or non-operational.

Before giving the final classification, reason step by step:
1. Explain what it means for an elevator to be operational.
2. Analyze what each LICENSESTATUS value implies.
3. Then provide the final classification with reasoning.

# *Output* :

### Step 1 — What does it mean for an elevator to be operational?

An elevator is **operational** if it holds a valid, legally recognized license that permits it to be used by building occupants or the public.

In Ontario, the TSSA (Technical Standards and Safety Authority) regulates elevating devices under the *Technical Standards and Safety Act*. For this classification, a device is considered operational when **all three of the following are true**:

1. **A license exists** — the device has a record in the TSSA licensing system.
2. **The license has not been revoked, cancelled, or permanently closed** — the regulatory relationship between the licenseholder and TSSA is still intact.
3. **The device can legally run or return to service without a new application** — temporary holds and in-progress renewals satisfy this condition; expired or cancelled statuses do not.

A device is **non-operational** if its license has lapsed, been formally cancelled, or been permanently terminated — meaning it cannot legally run until a new application is submitted and approved.

### Step 2 — What does each LICENSESTATUS value imply?

Analyzing each of the 11 distinct values against the definition above:

**ACTIVE** — The license is current and valid. The device has met all regulatory requirements and is fully permitted to run. No ambiguity.

**PENDING_RENEWAL** — The owner has initiated the renewal process before the license expired. The device retains its legal standing during this window. TSSA treats the device as licensed while the paperwork is processed; it has not been cancelled.

**BY REQUEST** — The licenseholder voluntarily suspended the license — a common action when a building closes temporarily for renovation or changes ownership. The license is not revoked; it is paused at the owner's direction. The device is still physically intact and can return to service when the hold is lifted. No new application is required.

**HOLD_TSD** — TSSA has placed an administrative hold on the license, typically while awaiting compliance documentation or resolving a procedural issue. The device has not been decommissioned; TSSA expects the hold to resolve and the license to return to ACTIVE. This is not a penalty or revocation — it is an administrative gate.

**ENTERED** — A data-entry or intake state. The record has been created in the TSSA system but has not yet been fully processed or approved. No rejection, cancellation, or termination has been recorded. The device cannot yet be confirmed operational, but it has not been ruled non-operational either. Treated as provisionally operational pending completion of the intake process.

**EXPIRED** — The license passed its expiry date without renewal. Under Ontario regulations, operating a device with an expired license is unlawful. The device must not run until the license is renewed or a new one issued. This is a lapse by inaction — recoverable if the owner acts, but non-operational in its current state.

**CANCELLED_NOT_RENEWED** — The owner did not renew the license after expiry, and TSSA formally closed it. This is a more finalized version of EXPIRED: the cancellation has been processed, and the license cannot simply be renewed — a new application may be required. The device is not legally permitted to operate.

**CANCELLED_BY_CUST_REQ** — The licenseholder explicitly requested cancellation. This typically occurs when a device is permanently removed from service, replaced, or a property is sold and the new owner will apply separately. The intent is clear: the device is out of service. The license relationship is intentionally ended.

**CANCELLED** — A general-purpose cancellation state. The specific reason is not encoded in this value, but the outcome is unambiguous: the license is no longer valid. The device cannot legally operate.

**TERMINATED** — A permanent revocation or decommissioning. Stronger than cancellation: the license relationship with TSSA has been ended definitively. This may result from regulatory enforcement, device removal, or other causes. The device is out of service and the license cannot be reinstated — a new application would be required.

**TERMINATED DECEASED** — The license was closed because the licenseholder died. The license cannot be transferred or continued under the deceased's name. The device cannot operate legally until a new licenseholder obtains a new license. A specific variant of termination, but functionally identical in outcome: no valid license exists.

### Step 3 — Final classification

Applying the definition from Step 1 to the analysis in Step 2:

| Status | Category | Key reasoning |
|---|---|---|
| **ACTIVE** | Operational | Current, valid license. Fully permitted to run. |
| **PENDING_RENEWAL** | Operational | License retained during the renewal window; not cancelled. |
| **BY REQUEST** | Operational | Voluntary hold by owner; license not revoked. Can return to service without a new application. |
| **HOLD_TSD** | Operational | Administrative hold by TSSA; not a revocation. Device expected to return to ACTIVE. |
| **ENTERED** | Operational | Intake state; no rejection or termination recorded. Provisionally operational. |
| **EXPIRED** | Non-operational | License lapsed by inaction. Operating the device would be unlawful under Ontario regulations. |
| **CANCELLED_NOT_RENEWED** | Non-operational | License formally closed after non-renewal. More finalized than EXPIRED; not legally permitted to operate. |
| **CANCELLED_BY_CUST_REQ** | Non-operational | Explicit cancellation by the licenseholder. Device intentionally taken out of service. |
| **CANCELLED** | Non-operational | License no longer valid. Device cannot legally operate regardless of reason for cancellation. |
| **TERMINATED** | Non-operational | License permanently revoked or device decommissioned. No path to reinstatement without a new application. |
| **TERMINATED DECEASED** | Non-operational | License closed due to licenseholder's death. No valid licenseholder; device cannot legally operate. |

### Summary

| Category | Statuses included | Record count | Share |
|---|---|---|---|
| **Operational** | ACTIVE, PENDING_RENEWAL, BY REQUEST, HOLD_TSD, ENTERED | 43,662 | 96.2% |
| **Non-operational** | EXPIRED, CANCELLED_NOT_RENEWED, CANCELLED_BY_CUST_REQ, CANCELLED, TERMINATED, TERMINATED DECEASED | 1,721 | 3.8% |

> **Chain-of-thought vs. zero-shot and few-shot:** All three prompting strategies produce the same final classification. The chain-of-thought approach adds the most value in the intermediate reasoning: by forcing explicit definition of "operational" before classifying individual statuses, it makes the classification boundary auditable. Edge cases like BY REQUEST, HOLD_TSD, and ENTERED are easier to justify — and easier to challenge — when the underlying criteria are stated explicitly rather than implied.

# TASK 2  – CALCULATION

# 2.1 Zero‑shot prompt

#  *Prompt* : 
Calculate the percentage of elevators with an expired license,
where LICENSEEXPIRYDATE is before today, based on the license dataset.

# *Output*: 

### Calculation

**Reference date:** 2026-05-06 (today)

**Method:** Parse the `LICENSEEXPIRYDATE` field for every record in `license.csv` and count records where the date is strictly before 2026-05-06.

| Metric | Value |
|---|---|
| Total records in license.csv | 45,383 |
| Records with LICENSEEXPIRYDATE before 2026-05-06 | 45,383 |
| Records with no expiry date | 0 |
| Records that could not be parsed | 0 |
| **Expired license percentage** | **100%** |

### Expiry date distribution

All dates in the dataset fall between 1977 and 2018. The most recent expiry date on record is **2018-05-12** — more than 8 years before today.

| Expiry year | Record count |
|---|---|
| 2017 | 40,183 |
| 2016 | 3,604 |
| 2018 | 4 |
| 2015 | 156 |
| 2014 | 130 |
| 2013 | 102 |
| 2012 | 85 |
| 2011 | 80 |
| 2010 | 54 |
| 2009 | 73 |
| 2008 and earlier | 912 |

### Interpretation

The result of **100%** is mathematically correct but reflects a data-context issue rather than a real operational problem:

- `license.csv` is a **historical snapshot** exported from the TSSA licensing system around 2017–2018, not a live feed.
- Because today's date (2026-05-06) is more than 8 years after the most recent stored expiry date, every record appears expired when compared to today.
- In a live system, only a fraction of licenses would be expired at any given moment. The `LICENSESTATUS` field encodes TSSA's actual classification: only **68 records** (0.15% of the fleet) carry `LICENSESTATUS = 'EXPIRED'`.

> **Recommendation for the dashboard:** Use `LICENSESTATUS` to identify expired licenses rather than date arithmetic on `LICENSEEXPIRYDATE`. The status field reflects TSSA's own classification at the time of export; the stored expiry dates are only meaningful relative to the original export date, not to today.

# 2.2 Few‑shot prompt

# *Prompt* : 
Calculate the percentage of elevators with an expired license.

Example:
If there are 100 elevators and 25 have LICENSEEXPIRYDATE before today,
the result is 25%.

Now calculate the percentage of expired licenses in the license dataset
and explain the result.

# *Output* : 

### Applying the example formula to the real dataset

The prompt provides the calculation pattern:

> If there are 100 elevators and 25 have LICENSEEXPIRYDATE before today, the result is 25%.

Generalizing the formula:

```
Expired (%) = (elevators with LICENSEEXPIRYDATE < today) / (total elevators) × 100
```

Substituting real values from `license.csv` (reference date: 2026-05-06):

```
Expired (%) = 45,383 / 45,383 × 100 = 100%
```

| Variable | Value |
|---|---|
| Total elevators in dataset | 45,383 |
| Elevators with LICENSEEXPIRYDATE before 2026-05-06 | 45,383 |
| **Expired license percentage** | **100%** |

### Explanation of the result

Following the same reasoning the example implies — that "expired" means the date has already passed — every single record in the dataset qualifies, giving a result of **100%**.

This is correct by the formula, but the result is driven by the age of the dataset rather than by a widespread compliance failure. The `license.csv` file is a historical export from the TSSA system; the most recent expiry date stored in the file is **2018-05-12**, which is more than 8 years before today. No matter which elevator is checked, its stored expiry date is in the past.

**Contrast with LICENSESTATUS:** The `LICENSESTATUS` field captures TSSA's own classification at the time of export. Only **68 records** (0.15%) carry the status `EXPIRED` — these are the elevators the regulator actually considered expired when the data was collected. The remaining records with past expiry dates were already renewed, cancelled, or reclassified before the export was taken.

> **Key takeaway from the few-shot approach:** The example anchors the formula clearly (count ÷ total × 100), which makes the calculation straightforward to apply. The more complex part — interpreting a 100% result — requires domain knowledge about the dataset's age that the example alone cannot supply.

# 2.3 Chain‑of‑thought prompt

# *Prompt* :

Calculate the percentage of elevators with expired licenses
(LICENSEEXPIRYDATE before today).

Reason step by step:
1. Determine how many elevators exist in total.
2. Determine how many licenses are expired.
3. Divide and calculate the percentage.
4. Explain the final result.

# *Output* : 

### Step 1 — Total elevators in the dataset

Count every record in `license.csv`. Each row represents one elevating device identified by a unique `ElevatingDevicesNumber`.

| | Count |
|---|---|
| Total records in license.csv | **45,383** |

---

### Step 2 — How many licenses are expired?

A license is expired when its `LICENSEEXPIRYDATE` is strictly before today (2026-05-06). Every date in the field was parsed; no records were missing a date and none failed to parse.

| | Count |
|---|---|
| Records with LICENSEEXPIRYDATE before 2026-05-06 | **45,383** |
| Records with no date / unparseable | 0 |

The earliest expiry date in the dataset is **1977-02-15**; the most recent is **2018-05-12**. Every single stored date predates today.

---

### Step 3 — Calculate the percentage

```
Expired (%) = expired records / total records × 100
Expired (%) = 45,383 / 45,383 × 100
Expired (%) = 100%
```

---

### Step 4 — Explain the result

**The formula produces 100%, which is correct by the arithmetic but misleading without context.**

The reason every record qualifies as expired is not that every elevator in Ontario has an invalid license today. It is that `license.csv` is a **historical data export** — a snapshot of the TSSA licensing system taken around 2017–2018. The most recent expiry date stored in the file is 2018-05-12, which is more than 8 years before today. No matter which record is examined, its stored expiry date is in the past.

To put it another way: the dataset was never updated after export, so the dates it contains describe the state of the fleet at export time, not today.

**What the data actually says about expiry at the time of export:**

The `LICENSESTATUS` field captures TSSA's own classification at the moment the data was collected. At that time:

| LICENSESTATUS | Records | Share |
|---|---|---|
| ACTIVE | 42,665 | 94.0% |
| EXPIRED | 68 | 0.15% |
| All other statuses | 2,650 | 5.85% |

Only 68 records (0.15%) were classified as `EXPIRED` by the regulator. The other 45,315 records had licenses that were either current, in renewal, or in a managed hold state — not expired — at the time of export.

> **Conclusion:** Date arithmetic on `LICENSEEXPIRYDATE` against today's date gives 100% because the dataset is 8+ years old. The operationally meaningful expired-license count is **68 (0.15%)**, derived from `LICENSESTATUS = 'EXPIRED'` — the field that reflects TSSA's actual classification.

##  TASK 3 – OPEN‑ENDED ANALYSIS

# 3.1 Zero-shot prompt

# *Prompt*: 
What patterns do you notice in the geographic distribution of elevators
across Ontario based on the license dataset?

# *Output*:

### Data source

Geographic information was extracted from the `LocationoftheElevatingDevice` field in `license.csv`. Each value follows the format `[STREET ADDRESS] [CITY] [POSTAL CODE] ON CA`. Cities were parsed from the token(s) immediately before the Canadian postal code, and Ontario's five postal regions were identified using the first letter of each postal code (M = GTA, L = Central, N = Southwestern, K = Eastern, P = Northern).

Total records analyzed: **45,383** (59 records could not be matched to a postal code and were excluded from regional counts).

---

### Pattern 1 — Extreme concentration in the Greater Toronto Area

The GTA (postal codes starting with M) holds **18,029 elevators — 39.8% of the entire Ontario fleet** — in a region that represents a small fraction of the province's land area. Toronto proper accounts for 12,134 devices on its own (26.7%).

If former Toronto municipalities — Scarborough, Etobicoke, North York, Willowdale, Don Mills, and Downsview, which were amalgamated into the City of Toronto in 1998 but still appear as separate city names in older records — are grouped with Toronto, the combined count reaches **16,747 (36.9%)** of the provincial fleet.

| Region | Elevators | Share |
|---|---|---|
| Greater Toronto Area (M) | 18,029 | 39.8% |
| Central Ontario (L) | 12,170 | 26.9% |
| Southwestern Ontario (N) | 6,694 | 14.8% |
| Eastern Ontario (K) | 6,491 | 14.3% |
| Northern Ontario (P) | 1,940 | 4.3% |

---

### Pattern 2 — Urban dominance: five cities hold nearly half the fleet

The five largest cities by elevator count account for **20,787 elevators (45.8%)** of the entire fleet:

| Rank | City | Elevators | Share of total |
|---|---|---|---|
| 1 | Toronto | 12,134 | 26.7% |
| 2 | Ottawa | 3,297 | 7.3% |
| 3 | Mississauga | 2,198 | 4.8% |
| 4 | London | 1,644 | 3.6% |
| 5 | Hamilton | 1,514 | 3.3% |
| — | **Top 5 total** | **20,787** | **45.8%** |

The remaining ~600+ cities and towns share the other 54.2% of the fleet — most individually holding fewer than 500 elevators.

---

### Pattern 3 — Secondary urban centres cluster around Toronto

The next tier of cities — Scarborough (1,458), Etobicoke (1,081), North York (912), Brampton (756), Markham (641), Oakville (533), Willowdale (524), Burlington (521), Richmond Hill (423) — are all part of or immediately adjacent to the Greater Toronto Area. This reinforces how strongly elevator density correlates with the Greater Toronto urban corridor.

---

### Pattern 4 — Northern Ontario is severely underrepresented

Northern Ontario (postal codes starting with P) covers more than **90% of Ontario's land area** but holds only **1,940 elevators (4.3%)** of the fleet. Thunder Bay (400) and Sudbury (373) are the two largest northern cities by elevator count. The pattern is consistent with population density: northern communities are smaller, more spread out, and have far fewer high-rise buildings requiring elevating devices.

---

### Pattern 5 — Pre-amalgamation city names persist in the dataset

Several city names that no longer exist as separate municipalities appear as distinct entries: Scarborough (1,458), Etobicoke (1,081), North York (912), Willowdale (524), Don Mills (334), and Downsview (304). These were amalgamated into the City of Toronto in 1998. Their presence as separate entries indicates that many device addresses were recorded before amalgamation and were never updated in the TSSA system — a data quality pattern worth noting for any address-based geographic analysis.

---

### Summary of patterns

| Pattern | Finding |
|---|---|
| Geographic concentration | 39.8% of the fleet is in the GTA — less than 1% of Ontario's land area |
| Urban dominance | The 5 largest cities hold 45.8% of all elevators |
| Toronto corridor | Most of the secondary cities are GTA-adjacent |
| Northern underrepresentation | 4.3% of elevators in 90%+ of Ontario's land area |
| Data quality | Pre-1998 municipality names persist, splitting the Toronto count across 7+ entries |

# 3.2 Few‑shot prompt

# *Prompt* : 
Analyze geographic patterns in elevator distribution.

Examples of possible observations:
- Higher concentration in major urban centers
- Fewer elevators in rural areas

Now analyze the license dataset and describe any geographic patterns you notice.

# *Output* : 


### Observations grounded in the prompt examples

The two examples in the prompt — urban concentration and rural scarcity — both hold strongly in the data. Below, each is confirmed with numbers, then extended with additional patterns the data reveals.

---

**Observation 1 (confirmed): Higher concentration in major urban centers**

Toronto alone accounts for **12,134 elevators — 26.7% of the entire provincial fleet** — making it by far the single largest concentration. The next four cities follow at a significant distance:

| City | Elevators | Share |
|---|---|---|
| Toronto | 12,134 | 26.7% |
| Ottawa | 3,297 | 7.3% |
| Mississauga | 2,198 | 4.8% |
| London | 1,644 | 3.6% |
| Hamilton | 1,514 | 3.3% |
| **Top 5 total** | **20,787** | **45.8%** |

Five cities hold nearly half the provincial fleet. The pattern is sharp: after the top 5, no single city exceeds 1,500 elevators.

---

**Observation 2 (confirmed): Fewer elevators in rural areas**

Northern Ontario — which covers more than 90% of Ontario's land area — holds only **1,940 elevators (4.3%)** of the fleet. Its two largest cities, Thunder Bay (400) and Sudbury (373), each hold fewer devices than many single Toronto neighbourhoods. Smaller northern communities typically have no high-rise buildings, making elevating devices rare outside of hospitals and municipal facilities.

---

**Observation 3 (extended): The GTA dominates at the regional level**

When grouped by Ontario's five postal regions, the Greater Toronto Area (postal codes starting with M) holds **18,029 elevators — 39.8%** of the fleet, nearly double any other region:

| Region | Elevators | Share |
|---|---|---|
| Greater Toronto Area | 18,029 | 39.8% |
| Central Ontario | 12,170 | 26.9% |
| Southwestern Ontario | 6,694 | 14.8% |
| Eastern Ontario | 6,491 | 14.3% |
| Northern Ontario | 1,940 | 4.3% |

The GTA-to-Northern ratio is approximately **9:1** — a stark disparity for regions that share the same province.

---

**Observation 4 (extended): A dense secondary cluster surrounds Toronto**

The cities immediately surrounding Toronto form a second tier of high elevator counts — all GTA-adjacent: Scarborough (1,458), Etobicoke (1,081), North York (912), Brampton (756), Markham (641), Oakville (533), Burlington (521), and Richmond Hill (423). Many of these are former municipalities absorbed into Toronto in 1998 that still appear as separate entries in the dataset. Combined with Toronto proper, the amalgamated city accounts for approximately **16,747 elevators (36.9%)**.

---

**Observation 5 (extended): Southwestern Ontario has a distinct secondary hub in Windsor**

Windsor (823 elevators) stands out as the dominant city in the far southwest — geographically isolated from the Toronto corridor and closer to Detroit than to Toronto. It represents a self-contained urban pocket, typical of Ontario's few mid-sized cities that are not part of the GTA commuter belt.

---

### Summary

| Observation | Evidence |
|---|---|
| Urban concentration | Top 5 cities = 45.8% of the fleet |
| Rural scarcity | Northern Ontario = 4.3% despite 90%+ of land area |
| GTA dominance | GTA region = 39.8%, nearly 9× Northern Ontario |
| Toronto corridor clustering | GTA-adjacent cities form a dense secondary ring |
| Isolated regional hub | Windsor anchors the far southwest independently |

> **Key takeaway from the few-shot approach:** The two example patterns (urban concentration, rural scarcity) were both confirmed and quantified. The few-shot examples provided the right frame — urban vs. rural — but the data revealed a more specific structure: it is not simply "cities vs. rural" but a gradient radiating outward from a single dominant core (Toronto), with a sharp drop-off beyond the GTA corridor.

# 3.3 Chain‑of‑thought prompt

# *Prompt* : 
Analyze the geographic distribution of elevators across Ontario.

Think step by step:
1. Identify where elevators are most concentrated.
2. Compare urban versus rural patterns.
3. Consider possible reasons for these patterns.
4. Summarize the main observations.

# *Output* : 


### Step 1 — Where are elevators most concentrated?

Geographic location was extracted from the `LocationoftheElevatingDevice` field in `license.csv`, using the city name embedded before each Canadian postal code. Postal code prefixes (first letter) were used to assign records to Ontario's five provincial regions.

**By region:**

| Region | Postal prefix | Elevators | Share |
|---|---|---|---|
| Greater Toronto Area | M | 18,029 | 39.8% |
| Central Ontario | L | 12,170 | 26.9% |
| Southwestern Ontario | N | 6,694 | 14.8% |
| Eastern Ontario | K | 6,491 | 14.3% |
| Northern Ontario | P | 1,940 | 4.3% |

**By city (top 10):**

| Rank | City | Elevators | Share |
|---|---|---|---|
| 1 | Toronto | 12,134 | 26.7% |
| 2 | Ottawa | 3,297 | 7.3% |
| 3 | Mississauga | 2,198 | 4.8% |
| 4 | London | 1,644 | 3.6% |
| 5 | Hamilton | 1,514 | 3.3% |
| 6 | Scarborough | 1,458 | 3.2% |
| 7 | Etobicoke | 1,081 | 2.4% |
| 8 | North York | 912 | 2.0% |
| 9 | Windsor | 823 | 1.8% |
| 10 | Brampton | 756 | 1.7% |

Concentration is highest in the GTA: Toronto alone holds 26.7% of the entire fleet, and the top 5 cities together hold 45.8%. The pattern narrows further — after rank 5, no single city exceeds 1,500 elevators.

---

### Step 2 — Urban versus rural patterns

**Urban Ontario** (the GTA, Central Ontario, and the major cities of Southwestern and Eastern Ontario) accounts for the overwhelming majority of the fleet. The 10 largest cities alone represent approximately **25,816 elevators — 56.9%** of all devices.

**Rural Ontario** presents a sharp contrast. Northern Ontario — the region with the largest land area by far, covering boreal forest, mining towns, and remote communities — holds only **1,940 elevators (4.3%)**. The GTA-to-Northern ratio is approximately **9:1**. Even within Northern Ontario, concentration is almost entirely in its two largest cities: Thunder Bay (400) and Sudbury (373) together account for 39.8% of the northern fleet; the remaining northern communities collectively hold fewer than 1,200 devices.

The urban–rural divide is not a gradual gradient — it is a sharp cliff. Once outside the GTA corridor and the handful of mid-size regional capitals (London, Hamilton, Kingston, Windsor), elevator counts drop rapidly to the tens or single digits per community.

---

### Step 3 — Possible reasons for these patterns

**Population density drives demand directly.**
Elevators are installed where buildings are tall enough to require them — typically structures of three or more storeys. High-rise residential towers, office buildings, hospitals, and shopping centres are overwhelmingly located in dense urban areas. Toronto's density (approximately 4,400 people/km²) generates demand for vertical transit that rural municipalities simply do not have.

**Economic activity concentrates in cities.**
Toronto is Canada's financial capital; Ottawa is the federal government hub; Mississauga hosts a major airport and industrial corridor; London and Hamilton are regional service centres. High economic activity produces large commercial and institutional buildings — the primary hosts of elevating devices.

**Infrastructure investment follows population and tax base.**
Municipalities with larger populations have more public buildings (city halls, libraries, transit stations, arenas) that require elevators under the Ontario Building Code's accessibility requirements. Smaller rural municipalities manage fewer such structures.

**The pre-1998 amalgamation effect inflates Toronto's apparent peers.**
Several cities in the top 10 — Scarborough (1,458), Etobicoke (1,081), North York (912) — are former municipalities absorbed into the City of Toronto in 1998. Their separate entries in the dataset reflect address records that predate amalgamation. If consolidated under Toronto, the true city total rises to approximately **16,747 (36.9%)**, making the gap between Toronto and every other city even more pronounced.

**Northern geography suppresses elevator demand structurally.**
Northern Ontario's communities are small (most under 10,000 residents), spread across vast distances, and economically centred on resource extraction rather than services or finance. These sectors do not generate the dense commercial building stock that drives elevator installations. The harsh climate and high construction costs further limit multi-storey development in smaller northern towns.

---

### Step 4 — Summary of main observations

| Observation | Key finding |
|---|---|
| **Single dominant core** | Toronto (26.7%) is the undisputed centre; Ottawa is a distant second at 7.3% |
| **GTA regional dominance** | The GTA holds 39.8% of the fleet — nearly 9× Northern Ontario's share |
| **Urban top-5 concentration** | 5 cities hold 45.8% of all elevators; the rest is distributed across 600+ communities |
| **Sharp urban–rural cliff** | Outside major cities, counts drop steeply; Northern Ontario = 4.3% of fleet in 90%+ of land area |
| **Toronto corridor cluster** | GTA-adjacent cities form a dense secondary ring (Mississauga, Brampton, Markham, Oakville…) |
| **Isolated regional hubs** | Windsor and Kingston anchor the southwest and southeast independently, outside the GTA orbit |
| **Data quality note** | Pre-1998 city names (Scarborough, Etobicoke, North York) split the Toronto count; true amalgamated Toronto = ~36.9% |

> **Chain-of-thought vs. zero-shot and few-shot:** All three approaches reach the same observations. The chain-of-thought structure adds the most value in Step 3: by forcing explicit reasoning about *why* the pattern exists — not just *what* the pattern is — it produces a more complete and defensible analysis. The few-shot examples anchored the urban/rural frame; the chain-of-thought unpacked the economic, demographic, and regulatory mechanisms behind it.

# Comparison of Prompting Techniques

## Task 1: Classification

Zero-shot produced a straightforward categorization of the 11 LICENSESTATUS values with clear reasoning. Few-shot improved clarity by anchoring the reasoning with concrete examples (ACTIVE, CANCELLED_NOT_RENEWED), which made ambiguous statuses like "BY REQUEST" easier to interpret. Chain-of-thought added explicit structure by first defining what "operational" means, then analyzing each value against that definition. The final classification was identical across all three approaches (5 operational, 6 non-operational), but chain-of-thought's step-by-step reasoning made the logic most transparent and verifiable.

## Task 2: Calculation

All three techniques reached the same result: 100% expired licenses. Zero-shot provided the calculation plus a data-age explanation and distribution table, showing that the dataset is a ~2017-2018 historical snapshot. Few-shot applied a formula from a worked example, which locked in consistency but required domain interpretation to explain why 100% doesn't reflect operational reality. Chain-of-thought broke the problem into discrete steps, making the calculation easier to audit and adding a useful final step that contrasted the 100% date-based metric with the 0.15% LICENSESTATUS=EXPIRED metric. For verification and confidence, chain-of-thought was most effective.

## Task 3: Open-ended Analysis

Zero-shot identified five geographic patterns and provided a clean regional breakdown with percentages. Few-shot confirmed those patterns using the urban/rural examples, then extended the analysis with the 9:1 GTA-to-North ratio and the Toronto corridor cluster observation. Chain-of-thought went further by structuring the analysis into four steps: regional concentration tables, urban vs. rural comparison (noting the "sharp cliff" rather than a gradient), then — uniquely — five reasons explaining *why* the distribution is so concentrated (population density, economic activity, infrastructure investment, pre-1998 amalgamation, Northern geography). This "why" step is where chain-of-thought added the most distinct value.

## Overall Observations

Chain-of-thought excels at complex reasoning and making verification straightforward, but added little value for Task 1's classification, where the answer was already clear. Few-shot is most useful when the desired reasoning pattern is easy to demonstrate by example; it anchored Task 1's ambiguous cases and scoped Task 3's observations. Zero-shot was surprisingly effective for all three tasks, providing solid baselines; the real gain came when structure (chain-of-thought) was needed for insight, or examples (few-shot) were needed for clarity on edge cases.